In [ ]:
# run this notebook after 8_analyze_twin_spectra_python
# use this notebook to call DNMs in trios
# after this, run 10_find_differences_siblings_python

In [ ]:
import hail as hl
import os
import pandas
from google.cloud import storage
import re
import numpy as np 

In [ ]:
bucket=os.getenv('WORKSPACE_BUCKET')
hl.init(idempotent=True)
hl.default_reference('GRCh38')


In [ ]:
for k in range(1,13):
    f_trios = f'{bucket}/data/trios_samples_{k}.txt'
    trios_all = pandas.read_table(f_trios, sep = ',')
    trios_all['par1'] = trios_all['par1'].astype(str)
    trios_all['par2'] = trios_all['par2'].astype(str)
    trios_all['off'] = trios_all['off'].astype(str)
    # keep only rows that do not appear in 'finished'
    trios_filtered = trios_all
    print(trios_filtered.shape)
    # read mt sub 
    mt_path =f'{bucket}/data/trios_samples_{k}.mt'
    mt = hl.read_matrix_table(mt_path)
    # now parse 
    for i in range(trios_filtered.shape[0]):
        par1 = trios_filtered["par1"].iloc[i]
        par2 = trios_filtered["par2"].iloc[i]
        off = trios_filtered["off"].iloc[i]
        parents_i = [str(par1), str(par2)]
        trio_i = [str(par1), str(par2), str(off)]
    
        # subset to the trio
        mt_subset_trio = mt.filter_cols(hl.literal(trio_i).contains(mt.s)) 
    
        # subset to high quality genotypes 
        mt_subset_trio = mt_subset_trio.annotate_entries(GT = hl.or_missing(mt_subset_trio.GQ >= 30, mt_subset_trio.GT))
        mt_subset_trio = mt_subset_trio.filter_rows(hl.agg.sum(hl.is_defined(mt_subset_trio.GT)) == 3, keep = True)

        # subset to those that have a passing FT 
        mt_subset_trio = mt_subset_trio.filter_rows(hl.agg.sum(mt_subset_trio.FT == "FAIL") == 0, keep = True)
    
        # subset to those where there is 1 het genotype  
        mt_subset_trio = mt_subset_trio.annotate_rows(is_het_ref = hl.agg.sum((mt_subset_trio.GT.is_het_ref()))) 
        mt_subset_trio = mt_subset_trio.filter_rows(mt_subset_trio.is_het_ref == 1, keep = True)
    
        # subset to those where there is 2 hom_ref genotypes
        mt_subset_trio = mt_subset_trio.annotate_rows(is_hom_ref = hl.agg.sum((mt_subset_trio.GT.is_hom_ref()))) 
        mt_subset_trio = mt_subset_trio.filter_rows(mt_subset_trio.is_hom_ref == 2, keep = True)
    
        # subset to parents 
        mt_subset_parents = mt_subset_trio.filter_cols(hl.literal(parents_i).contains(mt_subset_trio.s))
    
        # subset to where parents have 2 hom_ref genotypes 
        mt_subset_parents = mt_subset_parents.annotate_rows(is_hom_ref = hl.agg.sum((mt_subset_parents.GT.is_hom_ref()))) 
        mt_subset_parents = mt_subset_parents.filter_rows(mt_subset_parents.is_hom_ref == 2, keep = True)
    
        mt_subset_trio = mt_subset_trio.semi_join_rows(mt_subset_parents.rows())
        
        mt_fin = mt_subset_trio.repartition(50)
        mt_fin.GT.export(f'{bucket}/trios/difs/{par1}_{par2}_{off}_difs.tsv')
    

In [ ]:
for k in range(1,13):
    f_trios = f'{bucket}/data/trios_samples_{k}.txt'
    trios_all = pandas.read_table(f_trios, sep = ',')
    trios_all['par1'] = trios_all['par1'].astype(str)
    trios_all['par2'] = trios_all['par2'].astype(str)
    trios_all['off'] = trios_all['off'].astype(str)
    merged = trios_all.merge(finished, on=['par1', 'par2', 'off'], how='left', indicator=True)
    # keep only rows that do not appear in 'finished'
    trios_filtered = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
    print(trios_filtered.shape)
    # read mt sub 
    mt_path =f'{bucket}/data/trios_samples_{k}.mt'
    mt = hl.read_matrix_table(mt_path)
    # now parse 
    for i in range(trios_filtered.shape[0]):
        par1 = trios_filtered["par1"].iloc[i]
        par2 = trios_filtered["par2"].iloc[i]
        off = trios_filtered["off"].iloc[i]
        parents_i = [str(par1), str(par2)]
        trio_i = [str(par1), str(par2), str(off)]
    
        tsv_path = f'{bucket}/trios/difs/{par1}_{par2}_{off}_difs.tsv'
        t_in = hl.import_table(tsv_path, impute=True)
        
        split_vid = t_in.locus.split(":")
        t_in = t_in.annotate(
        locus = hl.locus(split_vid[0], hl.int(split_vid[1]), reference_genome='GRCh38'))
        t_loci = t_in.select('locus').key_by('locus').distinct()
        
        # subset to the trio
        mt_subset_trio = mt.filter_cols(hl.literal(trio_i).contains(mt.s)) 
        mt_subset_trio = mt_subset_trio.filter_rows(hl.is_defined(t_loci[mt_subset_trio.locus])) 
    
        # subset to high quality genotypes 
        # turn genotypes with gq < 30 into na
        mt_subset_trio = mt_subset_trio.annotate_entries(GT = hl.or_missing(mt_subset_trio.GQ >= 30, mt_subset_trio.GT))
        mt_subset_trio = mt_subset_trio.filter_rows(hl.agg.sum(hl.is_defined(mt_subset_trio.GT)) == 3, keep = True)

        # subset to those that have a passing FT 
        mt_subset_trio = mt_subset_trio.filter_rows(hl.agg.sum(mt_subset_trio.FT == "FAIL") == 0, keep = True)
    
        # subset to those where there is at least one nonref
        mt_subset_trio = mt_subset_trio.filter_rows(hl.agg.any(mt_subset_trio.GT.is_non_ref())) 

        mt_fin = mt_subset_trio.repartition(50)
        mt_fin.GT.export(f'{bucket}/trios/difs_all_alt/{par1}_{par2}_{off}_all_alt.tsv')
    

In [ ]:
%%bash 

mkdir -p trios 

gsutil -m cp -r $WORKSPACE_BUCKET/trios/difs ./trios/
gsutil -m cp -r $WORKSPACE_BUCKET/trios/difs_all_alt ./trios/

mkdir -p ./trios/formatted_difs